# 🎛️ Strata Studio

<a href="https://colab.research.google.com/github/PingPhantom/Colab/blob/main/nix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

จัดการโมเดล GGUF (10 ตัว + draft/MTP/N-gram) บน **llama.cpp** พร้อม **StrataRAG** และ UI 4 แท็บ
สำหรับ Google Colab GPU (T4 ฟรีก็ใช้ได้)

**วิธีใช้:** รันเซลล์ 1 (ติดตั้งสภาพแวดล้อม) → เซลล์ 2 (เปิด UI) แล้วใช้ผ่านแท็บได้เลย

> โค้ดจริงอยู่ในแพ็กเกจ [`strata/`](./strata) — notebook เหลือแค่เซลล์ setup สั้น ๆ


In [ ]:
# @title 1) ติดตั้งสภาพแวดล้อม (Nix + llama.cpp + uv) { display-mode: "form" }
import sys
import time
from pathlib import Path

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))  # ให้ import แพ็กเกจ strata/ ในรีโปนี้ได้

from strata.bootstrap import ensure_environment

t0 = time.perf_counter()
env = ensure_environment()
print(f"✅ ใช้เวลาเตรียม env {time.perf_counter() - t0:.1f}s • nixpkgs {env['nixpkgs'][:7]}")


In [ ]:
# @title 2) เปิด Studio UI (พอร์ต 18080) { display-mode: "form" }
from strata.notebook import launch, register_magics

register_magics()  # เปิดใช้ %%ruff ในเซลล์ใหม่ (ทางเลือก)
ui = launch()


## ใช้งานผ่านโค้ด (ไม่ต้องเปิด UI)

```python
from strata.download import install_model, remove_model
from strata.server import start_strata_server, stop_strata_server, get_openai_client

install_model("qwen35-9b-defiant-fable", auto_start_server=True)   # โหลด + สตาร์ท
client = get_openai_client()                                       # OpenAI-compatible ที่ :18080
print(client.chat.completions.create(
    model="qwen35-9b-defiant-fable",
    messages=[{"role": "user", "content": "สวัสดี"}],
).choices[0].message.content)
```

เรียก API จาก notebook/แอปอื่นได้ที่ `http://127.0.0.1:18080/v1` (OpenAI SDK, `curl`, LangChain ฯลฯ)

## RAG

```python
from strata.rag import GLOBAL_RAG

GLOBAL_RAG.add_files("/content/doc.pdf", chunk_size=500)
GLOBAL_RAG.save()          # /content/rag_store.npz (ชี้ไป Drive ได้)
print(GLOBAL_RAG.search("คำถาม", top_k=3))
```

## ตั้งค่าผ่าน environment variable

| ตัวแปร | ค่าเริ่มต้น | ใช้ทำอะไร |
| --- | --- | --- |
| `STRATA_PORT` | `18080` | พอร์ตของ API server/UI |
| `STRATA_HOME` | `/content` | โฟลเดอร์เก็บ log/คลัง RAG |
| `STRATA_MODEL_DIR` | `$STRATA_HOME/models` | ที่เก็บโมเดล GGUF |
| `STRATA_RAG_STORE` | `$STRATA_HOME/rag_store.npz` | ไฟล์คลัง RAG |
| `LLAMA_CPP_TAG` | release ล่าสุด | ปักหมุดเวอร์ชัน llama.cpp |
| `NIX_INSTALL_SHA256` | – | บังคับ checksum ของสคริปต์ติดตั้ง Nix |

รายละเอียดทั้งหมด (ข้อกำหนดเครื่อง, เทสต์, โครงสร้างโค้ด) อยู่ใน [`README.md`](./README.md)
